In [ ]:
import preprocess as pre
from pathlib import Path
import pandas as pd
import importlib.util
import anndata as ad
import scanpy as sc

In [ ]:
# Determine parent folder
base_dir = Path.cwd().parent
path = base_dir / "pipeline_paths_mnt.py"

# Import module dynamically
spec = importlib.util.spec_from_file_location("ppaths", path)
ppaths = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ppaths)

In [ ]:
# VARIABLES
# Quality control
min_cells = 200 
min_genes = 200
mito_threshold: float = 5.0,
min_cells_per_subject: int = 30

# Highly Variable Gene selection
nr_hvgs = 500

# train-test split
test_size: float = 0.2
random_state: int = 17

# to stratify AD+NCI v MCIP
stratify_key: str = 'label'

In [ ]:
# sets variables in file and folder names
run_vars = f'mg_{min_genes}_mc_{min_cells}_mhvg{nr_hvgs}' 

# to keep consistent paths
pp = ppaths.PipelinePaths("dev_local", full_pipeline_run_vars=run_vars)

In [ ]:
in_path = Path(pp.conv_data_path)

# datasets to include (keys are readable labels)
datasets = {}

datasets["test1"] = ad.read_h5ad(in_path / "test1.h5ad")
datasets["test2"] = ad.read_h5ad(in_path / "test2.h5ad")

In [ ]:
# get the order of the original genes (for verification after processing)
genes_ordered = datasets[list(datasets.keys())[0]].var_names.tolist()

In [ ]:
# create labels for AD, NCI and MCIP status, 
# based on cogdx, braaksc, and ceradc scores
for label, adata in datasets.items():
    datasets[label] = pre.create_new_labels(adata, drop_unknown=True)

In [ ]:
# save genes that exist in reactome to file
reactome_genes = pre.save_reactome_genes(pp.pathway_data_path, 
        filename_read='ReactomePathways.gmt', 
        filename_save='reactome_genes.txt'
        )

In [ ]:
# to inspect final objects
final_datasets = {}

for label, adata in datasets.items():
    # keep only genes that are in Reactome
    adata = pre.filter_reactome_genes(adata, set(reactome_genes))

    ## Quality Control
    # remove cells with less than n expressed genes
    adata = pre.filter_cells_by_min_genes(adata, min_genes=min_genes)

    # filter genes to keep only genes expressed in at least n cells
    adata = pre.filter_genes_by_min_expression(adata, min_cells=min_cells)

    # remove cells with mito gene expression over threshold
    adata = pre.filter_cells_by_mitochondrial_content(adata, mt_threshold=mito_threshold)

    # filter genes to keep only subjects with at least n cells
    adata = pre.filter_low_cell_subjects(adata, min_cells_per_subject=min_cells_per_subject)

    datasets[label] = adata
    
    # Stratify AD+NCI, MCIP
    adata, mcip_adata = pre.separate_stratified_data(
            adata,
            save_dir=pp.stratified_path,
            file_prefix=[f"{label}_AD_NCI", f"{label}_MCIP"]
        )

    # Split AD+NCI into train/test
    train_subjects, test_subjects = pre.train_test_split_subjects(
            adata,
            test_size=test_size,
            random_state=random_state,
            stratify_key=stratify_key,
            save_dir=pp.stratified_path,
            file_prefix=label
        )

    # Split the AnnData object into training and testing sets based on the provided subject lists
    train_adata, test_adata = pre.train_test_split_adata(
        adata,
        train_subjects=train_subjects,
        test_subjects=test_subjects,
        save_dir=pp.stratified_path,
        file_prefix=label
    )

    # HVG selection from train set
    hvgs = pre.select_hvgs(train_adata, nr_hvgs)

    # Filter hvgs
    train_adata = pre.filter_hvgs(train_adata, hvgs)
    test_adata = pre.filter_hvgs(test_adata, hvgs)
    mcip_adata = pre.filter_hvgs(mcip_adata, hvgs)

    # pseudobulk
    # sum counts per subject and high res cell type
    train_adata = pre.aggregate_to_pseudobulk(
        train_adata, 
        bulk_by='mean', 
        )
    
    test_adata = pre.aggregate_to_pseudobulk(
        test_adata, 
        bulk_by='mean', 
        )
    
    mcip_adata = pre.aggregate_to_pseudobulk(
        mcip_adata, 
        bulk_by='mean', 
        )

    # standardize train set, apply transformation to test and MCIP sets
    train_adata_standardized, scaler = pre.standardize(train_adata)
    test_adata_standardized = pre.standardize(test_adata, scaler)
    mcip_adata_standardized = pre.standardize(mcip_adata, scaler)

    # Save the standardized training, testing and MCIP sets to disk as pandas DataFrames
    train_adata_standardized.to_df().reset_index().to_csv(
        pp.compl_full_pipe_path / f'{label}_train_preprocessed.csv', 
        index=False)
    
    test_adata_standardized.to_df().reset_index().to_csv(
        pp.compl_full_pipe_path / f'{label}_test_preprocessed.csv', 
        index=False)
    
    mcip_adata_standardized.to_df().reset_index().to_csv(
        pp.compl_full_pipe_path / f'{label}_mcip_preprocessed.csv', 
        index=False)

    # Save the standardized training, testing and MCIP sets to disk as anndata files
    sc.write(pp.compl_full_pipe_path / f'{label}_train_preprocessed.h5ad', 
             train_adata_standardized)
    sc.write(pp.compl_full_pipe_path / f'{label}_test_preprocessed.h5ad', 
             test_adata_standardized)
    sc.write(pp.compl_full_pipe_path / f'{label}_mcip_preprocessed.h5ad', 
             mcip_adata_standardized)

    final_datasets[f'train_{label}'] = train_adata_standardized
    final_datasets[f'test_{label}'] = test_adata_standardized
    final_datasets[f'mcip_{label}'] = mcip_adata_standardized

In [ ]:
# verify genes in correct order
verify = {'train': train_adata, 'test': test_adata, 'mcip':mcip_adata}
pre.verify_gene_order(verify, genes_ordered)

In [ ]:
for label, adata in final_datasets.items():
    print(adata)
